# 10 · Zheng & Fattahi (2026) — End-to-End Workflow, Depth Sensitivity & Multilook Noise

Reproduces and extends key diagnostic analyses from:
> **Zheng, Y., & Fattahi, H. (2026).** *Modeling, prediction, and retrieval of surface soil moisture from InSAR closure phase.* Remote Sensing of Environment, 333, 115104.


In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from navasar.slc import simulate_slc, multilook_interferogram
from navasar.closure import closure_phase_timeseries, _bw1_timeseries, _fullnet_timeseries
from navasar.dielectric import kz_soil
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
RNG = np.random.default_rng(42)


## 1. Fig. 7: End-to-End InSAR Simulation Pipeline


In [ ]:
T = 20
t_idx = np.arange(T)
BASE_KW = dict(n_pixels=400, n_layers=200, max_depth=0.30, sand=0.51, clay=0.13)
mv_base, mv_rain = 0.05, 0.20
mv_ts = np.full(T, mv_base)
mv_ts[8:12] = mv_rain
slc_uniform = simulate_slc(mv_ts, freq_ghz=1.4, rng=np.random.default_rng(42), sigma_profile='uniform', **BASE_KW)
ifg_mat = np.zeros((T, T), dtype=complex)
for i in range(T):
    for j in range(i + 1, T):
        ifg_mat[i, j] = multilook_interferogram(slc_uniform[i], slc_uniform[j])
        ifg_mat[j, i] = np.conj(ifg_mat[i, j])
bw1_ts  = np.rad2deg(_bw1_timeseries(ifg_mat))
full_ts = np.rad2deg(_fullnet_timeseries(ifg_mat))
cp_ts   = bw1_ts - full_ts
amp_ts  = np.abs(slc_uniform).mean(axis=1)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(t_idx, mv_ts, 'k-o', ms=5)
axes[0, 0].axvspan(7.5, 11.5, color='lightblue', alpha=0.4, label='Rain Event')
axes[0, 0].set_xlabel('Acquisition Index t')
axes[0, 0].set_ylabel('Moisture mv [m3/m3]')
axes[0, 0].set_title('(a) Soil Moisture Input')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
axes[0, 1].plot(t_idx, amp_ts, 'b-s', ms=5)
axes[0, 1].axvspan(7.5, 11.5, color='lightblue', alpha=0.4)
axes[0, 1].set_xlabel('Acquisition Index t')
axes[0, 1].set_ylabel('Mean SLC Amplitude [a.u.]')
axes[0, 1].set_title('(b) Mean SLC Amplitude')
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(t_idx, bw1_ts, 'b-o', ms=4, label='BW-1 (Short Baselines)')
axes[1, 0].plot(t_idx, full_ts, 'r--d', ms=4, label='Full-Network LS')
axes[1, 0].axvspan(7.5, 11.5, color='lightblue', alpha=0.4)
axes[1, 0].axhline(0, color='k', lw=0.8, ls=':')
axes[1, 0].set_xlabel('Acquisition Index t')
axes[1, 0].set_ylabel('Phase [deg]')
axes[1, 0].set_title('(c) BW-1 vs Full-Network InSAR')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()
axes[1, 1].plot(t_idx, cp_ts, 'g-^', ms=5, lw=1.5)
axes[1, 1].axvspan(7.5, 11.5, color='lightblue', alpha=0.4)
axes[1, 1].axhline(0, color='k', lw=0.8, ls=':')
axes[1, 1].set_xlabel('Acquisition Index t')
axes[1, 1].set_ylabel('Closure Phase [deg]')
axes[1, 1].set_title('(d) Closure Phase Step')
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig07_zheng_workflow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak Closure Phase Step: {cp_ts.max() - cp_ts.min():.2f} deg')


## 2. Fig. 7b: Effect of Vertical Scatterer Profiles


In [ ]:
scale_depths = [0.05, 0.10, 0.15, 0.25]
plt.figure(figsize=(10, 5))
plt.plot(t_idx, cp_ts, 'k-', lw=2, label='Uniform profile')
for sd in scale_depths:
    slc_exp = simulate_slc(mv_ts, freq_ghz=1.4, rng=np.random.default_rng(42), sigma_profile='exponential', scale_depth=sd, **BASE_KW)
    cp_exp, _, _ = closure_phase_timeseries(slc_exp)
    plt.plot(t_idx, np.rad2deg(cp_exp), '--o', ms=4, label=f'Exponential (scale={sd*100:.0f} cm)')
plt.axvspan(7.5, 11.5, color='lightblue', alpha=0.3, label='Rain Event')
plt.xlabel('Acquisition Index t')
plt.ylabel('Closure Phase [deg]')
plt.title('Fig. 7b — Effect of Scatterer Profile on Closure Phase Step')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../examples/fig07b_profile_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Fig. 9: Sensitivity to Maximum Penetration Depth D


In [ ]:
depths = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.45, 0.60])
cp_steps = []
for D in depths:
    slc_d = simulate_slc(mv_ts, freq_ghz=1.4, rng=np.random.default_rng(42), n_pixels=300, n_layers=200, max_depth=D, sand=0.51, clay=0.13)
    cp_d, _, _ = closure_phase_timeseries(slc_d)
    cp_steps.append(cp_d.max() - cp_d.min())
plt.figure(figsize=(8, 5))
plt.plot(depths * 100, cp_steps, 'ro-', lw=1.8, ms=6)
plt.xlabel('Max Integration Depth D [cm]')
plt.ylabel('Closure Phase Step Magnitude [deg]')
plt.title('Fig. 9 — Closure Phase Step Sensitivity to Depth D')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig09_zheng_depth_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Fig. 10: Multilooking and Noise Floor Convergence


In [ ]:
looks_list = [25, 50, 100, 200, 400, 800]
n_trials = 6
mean_steps, std_steps = [], []
for M in looks_list:
    trial_steps = []
    for seed in range(n_trials):
        slc_m = simulate_slc(mv_ts, freq_ghz=1.4, rng=np.random.default_rng(100 + seed), n_pixels=M, n_layers=100, max_depth=0.25, sand=0.51, clay=0.13)
        cp_m, _, _ = closure_phase_timeseries(slc_m)
        trial_steps.append(cp_m.max() - cp_m.min())
    mean_steps.append(np.mean(trial_steps))
    std_steps.append(np.std(trial_steps))
mean_steps, std_steps = np.array(mean_steps), np.array(std_steps)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.errorbar(looks_list, mean_steps, yerr=std_steps, fmt='bo-', capsize=4, lw=1.5)
ax1.set_xscale('log')
ax1.set_xlabel('Number of Looks M (pixels)')
ax1.set_ylabel('Retrieved Closure Phase Step [deg]')
ax1.set_title('Step Convergence vs Number of Looks')
ax1.grid(True, alpha=0.3)
ax2.loglog(looks_list, std_steps, 'r-^', ms=6, label='Empirical std')
ref_curve = std_steps[0] * np.sqrt(looks_list[0] / np.array(looks_list))
ax2.loglog(looks_list, ref_curve, 'k--', label='Theoretical 1/sqrt(M) Rate')
ax2.set_xlabel('Number of Looks M')
ax2.set_ylabel('Std Dev [deg]')
ax2.set_title('Fig. 10 — Closure Phase Noise Floor Decay')
ax2.grid(True, which='both', alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.savefig('../examples/fig10_zheng_multilook_noise.png', dpi=150, bbox_inches='tight')
plt.show()
